# 01. Chuẩn hóa dữ liệu thô

Notebook này gộp 2 file dữ liệu gốc (JSONL), gọi GitHub API để lấy ngày commit của từng đoạn comment AI, rồi lưu thành một file CSV chuẩn cho các bước sau.

Import các thư viện cần thiết để đọc dữ liệu, gọi API và chạy song song.

In [1]:
import json
import os
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
import pandas as pd
import requests

Đọc 2 file JSONL gốc và gộp lại theo cột `match_id` để giữ đầy đủ thông tin về lần thay đổi đầu tiên (nếu có).

In [2]:
DATASET_PATH = "../data/raw/comment_codeblock_dataset.jsonl"
DATASET_WITH_CHANGE_PATH = "../data/raw/comment_codeblock_first_change_dataset.jsonl"

full_dataset = pd.read_json(DATASET_PATH, lines=True)
dataset_with_change = pd.read_json(DATASET_WITH_CHANGE_PATH, lines=True)

dataset = full_dataset.merge(
    dataset_with_change,
    on="match_id",
    how="left",
    suffixes=("", "_drop")
)
drop_cols = [c for c in dataset.columns if c.endswith("_drop")]
dataset = dataset.drop(columns=drop_cols)

print(f"Loaded {len(dataset):,} rows and {dataset.shape[1]} columns")
dataset.head(3)

Loaded 35,278 rows and 12 columns


,match_id,trace_id,repo_full_name,path,language,file_url,comment,code_block,first_change_commit_message,first_change_commit_oid,first_change_commit_url,first_change_date
0,python__written_by_gpt__433861003__e397ad21a6e...,python__written_by_gpt__433861003__e397ad21a6e...,flowerett/aoc,2022/python/d1-gpt.py,python,https://github.com/flowerett/aoc/blob/71574e59...,# solution written by GPT chatbot\n# works and...,# solution written by GPT chatbot\n# works and...,NaN,NaN,NaN,NaN
1,python__written_by_gpt__433861003__e397ad21a6e...,python__written_by_gpt__433861003__e397ad21a6e...,flowerett/aoc,2022/python/d1-gpt.py,python,https://github.com/flowerett/aoc/blob/71574e59...,# solution written by GPT chatbot\n# works and...,# solution written by GPT chatbot\n# works and...,NaN,NaN,NaN,NaN
2,python__written_by_ai__437874902__14f89ee997ec...,python__written_by_ai__437874902__14f89ee997ec...,tipa16384/adventofcode,2017/puzzle2.py,python,https://github.com/tipa16384/adventofcode/blob...,# Written by AI\n,# Written by AI\n\ndef spreadsheet_checksum(li...,NaN,NaN,NaN,NaN


Thiết lập phiên kết nối tới GitHub API bằng biến môi trường `GITHUB_TOKEN`.

In [3]:
TOKEN = os.getenv("GITHUB_TOKEN")
if not TOKEN:
    raise RuntimeError("Please set the GITHUB_TOKEN environment variable before running this cell.")

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
})

Tạo hàm truy vấn GraphQL theo từng nhóm file để lấy nội dung file và lịch sử git blame cùng lúc, giúp giảm số lần gọi API.

In [4]:
def fetch_graphql_batch(batch):
    query_parts = []
    for idx, (owner, repo, sha, file_path) in enumerate(batch):
        owner_json = json.dumps(owner)
        repo_json = json.dumps(repo)
        sha_json = json.dumps(sha)
        path_json = json.dumps(file_path)
        expr_json = json.dumps(f"{sha}:{file_path}")

        query_parts.append(
            f'c{idx}: repository(owner: {owner_json}, name: {repo_json}) {{ '
            f'  commit: object(expression: {sha_json}) {{ '
            f'    ... on Commit {{ '
            f'      blame(path: {path_json}) {{ '
            f'        ranges {{ startingLine endingLine commit {{ oid committedDate }} }} '
            f'      }} '
            f'    }} '
            f'  }} '
            f'  blob: object(expression: {expr_json}) {{ '
            f'    ... on Blob {{ text }} '
            f'  }} '
            f'}}'
        )

    query = f"query {{ {' '.join(query_parts)} }}"
    try:
        r = session.post("https://api.github.com/graphql", json={"query": query}, timeout=30)
        if r.status_code != 200:
            return [(None, None)] * len(batch)

        data = r.json().get("data", {}) or {}
        results = []
        for idx in range(len(batch)):
            repo_data = data.get(f"c{idx}") or {}
            commit_obj = repo_data.get("commit") or {}
            blob_obj = repo_data.get("blob") or {}
            blame = commit_obj.get("blame") or {}
            ranges = blame.get("ranges") or []
            blob_text = blob_obj.get("text")
            results.append((ranges, blob_text))
        return results
    except requests.RequestException:
        return [(None, None)] * len(batch)

Tìm vị trí dòng bắt đầu và kết thúc của đoạn comment AI trong nội dung file vừa lấy về.

In [5]:
def find_comment_line_span(file_text, comment, code_block=None):
    if not file_text or not isinstance(file_text, str):
        return None, None

    file_lines = file_text.splitlines()
    file_lines_stripped = [l.strip() for l in file_lines]
    total_lines = len(file_lines)
    if total_lines == 0:
        return None, None

    comment_lines = (
        [l.strip() for l in comment.splitlines() if l.strip()]
        if isinstance(comment, str) else []
    )

    if isinstance(comment, str) and comment.strip():
        raw_c = comment.strip()
        pos = file_text.find(raw_c)
        if pos != -1:
            start = file_text[:pos].count("\n") + 1
            end = start + raw_c.count("\n")
            return start, end

    if comment_lines:
        n = len(comment_lines)
        for i in range(total_lines - n + 1):
            if file_lines_stripped[i : i + n] == comment_lines:
                return i + 1, i + n

    substantive = [l for l in comment_lines if l.strip('"' + "'/*#-").strip() != ""]
    if substantive and len(substantive) != len(comment_lines):
        n_sub = len(substantive)
        for i in range(total_lines):
            sub_idx = 0
            match_start = None
            match_end = None
            for j in range(i, min(i + n_sub + 5, total_lines)):
                f_line = file_lines_stripped[j]
                if not f_line or f_line in ('"' * 3, "'" * 3, "/*", "*/", "*", "//", "#"):
                    continue
                if f_line == substantive[sub_idx] or substantive[sub_idx] in f_line:
                    if match_start is None:
                        match_start = j + 1
                    match_end = j + 1
                    sub_idx += 1
                    if sub_idx == n_sub:
                        return match_start, match_end
                else:
                    break

    ai_keywords = ["chatgpt", "gpt", "copilot", "ai", "generated", "written", "assisted", "prompt"]
    ai_target = None
    for l in substantive:
        if any(k in l.lower() for k in ai_keywords):
            ai_target = l
            break
    if ai_target:
        norm_target = " ".join(ai_target.lower().split())
        for idx, line in enumerate(file_lines, start=1):
            if ai_target in line or norm_target in " ".join(line.lower().split()):
                return idx, idx

    if isinstance(code_block, str) and code_block.strip():
        cb_lines = [l.strip() for l in code_block.splitlines() if l.strip()]
        for l in cb_lines:
            if any(k in l.lower() for k in ai_keywords):
                norm_l = " ".join(l.lower().split())
                for idx, line in enumerate(file_lines, start=1):
                    if l in line or norm_l in " ".join(line.lower().split()):
                        return idx, idx

    return None, None

Đối chiếu vị trí dòng comment với dữ liệu git blame để lấy mã commit và ngày commit tương ứng.

In [6]:
def resolve_blame_commit(ranges, file_text, start_line, end_line):
    if not ranges or start_line is None or end_line is None:
        return None, None

    file_lines = file_text.splitlines() if file_text else []
    ai_keywords = ["chatgpt", "gpt", "copilot", "ai", "generated", "written", "assisted", "prompt"]

    ai_line_target = None
    for line_no in range(start_line, end_line + 1):
        if 1 <= line_no <= len(file_lines):
            line_content = file_lines[line_no - 1].lower()
            if any(k in line_content for k in ai_keywords):
                ai_line_target = line_no
                break

    target_line = ai_line_target if ai_line_target is not None else start_line

    for r in ranges:
        s = r.get("startingLine", 0)
        e = r.get("endingLine", 0)
        if s <= target_line <= e:
            c = r.get("commit") or {}
            return c.get("oid"), c.get("committedDate")

    for line_no in range(start_line, end_line + 1):
        for r in ranges:
            if r.get("startingLine", 0) <= line_no <= r.get("endingLine", 0):
                c = r.get("commit") or {}
                return c.get("oid"), c.get("committedDate")

    return None, None

Chạy truy vấn song song cho toàn bộ các file duy nhất và ghép đường dẫn commit cùng ngày commit vào từng dòng dữ liệu.

In [7]:
def get_all_commit_info(file_urls, file_paths, comments, code_blocks, batch_size=15):
    parsed_all = []
    for url, path, comm, blk in zip(file_urls, file_paths, comments, code_blocks):
        if pd.isna(url) or pd.isna(path):
            parsed_all.append((None, None, None, None, None, None))
            continue
        parts = str(url).split("/")
        if len(parts) >= 7:
            parsed_all.append((parts[3], parts[4], parts[6], str(path), str(comm), str(blk)))
        else:
            parsed_all.append((None, None, None, None, None, None))

    unique_files = list(dict.fromkeys(
        (item[0], item[1], item[2], item[3]) for item in parsed_all if item[0] is not None
    ))
    batches = [unique_files[i : i + batch_size] for i in range(0, len(unique_files), batch_size)]

    file_data_map = {}
    with ThreadPoolExecutor(max_workers=6) as executor:
        for batch, batch_results in zip(batches, executor.map(fetch_graphql_batch, batches)):
            for key, (ranges, text) in zip(batch, batch_results):
                file_data_map[key] = (ranges, text)

    commit_urls = []
    commit_dates = []
    for item in parsed_all:
        if item[0] is None:
            commit_urls.append(None)
            commit_dates.append(None)
            continue
        owner, repo, sha, path, comment, blk = item
        ranges, text = file_data_map.get((owner, repo, sha, path), (None, None))
        start_line, end_line = find_comment_line_span(text, comment, blk)
        oid, raw_date = resolve_blame_commit(ranges, text, start_line, end_line)

        if oid:
            commit_urls.append(f"https://github.com/{owner}/{repo}/commit/{oid}")
        else:
            commit_urls.append(None)

        formatted_date = (
            pd.to_datetime(raw_date, utc=True).strftime("%Y-%m-%dT%H:%M:%SZ")
            if raw_date
            else None
        )
        commit_dates.append(formatted_date)

    return commit_urls, commit_dates

real_commit_urls, real_commit_dates = get_all_commit_info(
    dataset["file_url"], dataset["path"], dataset["comment"], dataset["code_block"]
)

Tạo bảng dữ liệu chuẩn. Giữ lại các cột `comment`, `file_url` và `change_commit_message` để phục vụ phân tích phía sau, đồng thời giữ nguyên ký tự xuống dòng để đếm số dòng code chính xác.

In [8]:
output = pd.DataFrame({
    "id": dataset["match_id"],
    "repo": dataset["repo_full_name"],
    "language": dataset["language"],
    "file_url": dataset["file_url"],
    "commit_url": real_commit_urls,
    "file": dataset["path"],
    "comment": dataset["comment"],
    "code_block": dataset["code_block"],
    "commit_date": real_commit_dates,
    "change_commit_url": dataset["first_change_commit_url"],
    "change_commit_date": dataset["first_change_date"],
    "change_commit_message": dataset["first_change_commit_message"],
})

print(f"Output shape: {output.shape}")
output.head(3)

Output shape: (35278, 12)


,id,repo,language,file_url,commit_url,file,comment,code_block,commit_date,change_commit_url,change_commit_date,change_commit_message
0,python__written_by_gpt__433861003__e397ad21a6e...,flowerett/aoc,python,https://github.com/flowerett/aoc/blob/71574e59...,https://github.com/flowerett/aoc/commit/52a0fb...,2022/python/d1-gpt.py,# solution written by GPT chatbot\n# works and...,# solution written by GPT chatbot\n# works and...,2022-12-03T08:41:16Z,NaN,NaN,NaN
1,python__written_by_gpt__433861003__e397ad21a6e...,flowerett/aoc,python,https://github.com/flowerett/aoc/blob/71574e59...,https://github.com/flowerett/aoc/commit/52a0fb...,2022/python/d1-gpt.py,# solution written by GPT chatbot\n# works and...,# solution written by GPT chatbot\n# works and...,2022-12-03T08:41:16Z,NaN,NaN,NaN
2,python__written_by_ai__437874902__14f89ee997ec...,tipa16384/adventofcode,python,https://github.com/tipa16384/adventofcode/blob...,https://github.com/tipa16384/adventofcode/comm...,2017/puzzle2.py,# Written by AI\n,# Written by AI\n\ndef spreadsheet_checksum(li...,2022-12-04T05:58:44Z,NaN,NaN,NaN


Lưu file CSV bản mới nhất và một bản sao lưu theo ngày giờ vào thư mục `data/interim/`.

In [9]:
INTERIM_DIR = "../data/interim"
os.makedirs(INTERIM_DIR, exist_ok=True)

timestamp = datetime.today().strftime("%Y-%m-%d_%H-%M-%S")
output.to_csv(f"{INTERIM_DIR}/standardized_data_{timestamp}.csv", index=False)
output.to_csv(f"{INTERIM_DIR}/latest_standardized_data.csv", index=False)
print("Saved standardized CSV files.")

Saved standardized CSV files.
